In [1]:
import json, copy
from collections import Counter

IN_PATH = "unibench_skeleton.jsonl"
OUT_PATH = "unibench_skeleton.fixed_ABC.jsonl"

# --- 여기 있는 ID들은 "C가 유령" 케이스: C를 제거하고 A,B만 유지 ---
DROP_C_IDS = {
    "unibench_UniBench_English_short_Reddit_id_70",
    "unibench_UniBench_English_short_Reddit_id_66",
    "unibench_UniBench_English_short_short_id_109_DIT_Politeness",
    "unibench_UniBench_English_long_Reddit_id_18",
    "unibench_UniBench_English_long_long_id_54_MJI_Politeness",
    "unibench_UniBench_English_long_long_id_91_DIT_Relationships",
    "long_id_96_DIT_Legality",
    "unibench_UniBench_English_short_withDemo_Reddit_id_70",
    "unibench_UniBench_English_short_withDemo_Reddit_id_66",
    "unibench_UniBench_English_short_withDemo_short_id_109_DIT_Politeness",
    "unibench_UniBench_English_long_withDemo_Reddit_id_18",
    "unibench_UniBench_English_long_withDemo_long_id_54_MJI_Politeness",
    "unibench_UniBench_English_long_withDemo_long_id_91_DIT_Relationships",
    "unibench_UniBench_English_long_withDemo_long_id_96_DIT_Legality",
}

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def get_ex_id(ex):
    return ex.get("id") or ex.get("instance_id") or ""

def repair_abc_instance(ex):
    """
    Only touches labels/options B/C (A is never edited):
    If label_space == ["A","B","C"]:
      - If id in DROP_C_IDS: drop C, keep A,B
      - Else: drop B, rename C -> B  (your original rule)
    Also updates meta labels accordingly (only B/C changes).
    """
    ls = ex.get("label_space")
    if ls != ["A", "B", "C"]:
        return ex, False, ""

    ex_id = get_ex_id(ex)
    drop_c = (ex_id in DROP_C_IDS)

    new_ex = copy.deepcopy(ex)

    bundle = new_ex.get("bundle", {})
    items = bundle.get("items", [])
    if not isinstance(items, list):
        return ex, False, "items_not_list"

    # quick presence checks
    has_A = any(isinstance(it, dict) and it.get("item_id") == "A" for it in items)
    has_B = any(isinstance(it, dict) and it.get("item_id") == "B" for it in items)
    has_C = any(isinstance(it, dict) and it.get("item_id") == "C" for it in items)

    if not has_A:
        return ex, False, "missing_A"

    if drop_c:
        # require real B to keep
        if not has_B:
            return ex, False, "missing_B_for_dropC"

        new_items = []
        for it in items:
            if not isinstance(it, dict):
                continue
            iid = it.get("item_id")
            if iid == "C":
                continue  # drop C
            new_items.append(it)

        bundle["items"] = new_items
        new_ex["bundle"] = bundle
        new_ex["label_space"] = ["A", "B"]

        # meta adjustments: drop C, keep B
        meta = new_ex.get("meta")
        if isinstance(meta, dict):
            meta = dict(meta)
            sd = meta.get("selection_distribution")
            if isinstance(sd, dict):
                sd2 = {}
                for k, v in sd.items():
                    if k == "C":
                        continue
                    sd2[k] = v
                meta["selection_distribution"] = sd2

            anns = meta.get("annotations")
            if isinstance(anns, list):
                new_anns = []
                for a in anns:
                    if not isinstance(a, dict):
                        new_anns.append(a); continue
                    a2 = dict(a)
                    if a2.get("selected_action") == "C":
                        a2["selected_action"] = None  # ghost
                    new_anns.append(a2)
                meta["annotations"] = new_anns
            new_ex["meta"] = meta

        return new_ex, True, "drop_C_keep_AB"

    else:
        # default: drop B, rename C -> B
        if not has_C:
            return ex, False, "missing_C_for_dropB"

        new_items = []
        for it in items:
            if not isinstance(it, dict):
                continue
            iid = it.get("item_id")
            if iid == "B":
                continue  # drop B
            if iid == "C":
                it2 = dict(it)
                it2["item_id"] = "B"  # C -> B
                new_items.append(it2)
            else:
                new_items.append(it)

        bundle["items"] = new_items
        new_ex["bundle"] = bundle
        new_ex["label_space"] = ["A", "B"]

        # meta adjustments: drop B, C->B
        meta = new_ex.get("meta")
        if isinstance(meta, dict):
            meta = dict(meta)
            sd = meta.get("selection_distribution")
            if isinstance(sd, dict):
                sd2 = {}
                for k, v in sd.items():
                    if k == "A":
                        sd2["A"] = v
                    elif k == "C":
                        sd2["B"] = v
                    elif k == "B":
                        continue  # drop ghost
                    else:
                        sd2[k] = v
                meta["selection_distribution"] = sd2

            anns = meta.get("annotations")
            if isinstance(anns, list):
                new_anns = []
                for a in anns:
                    if not isinstance(a, dict):
                        new_anns.append(a); continue
                    a2 = dict(a)
                    if a2.get("selected_action") == "C":
                        a2["selected_action"] = "B"
                    elif a2.get("selected_action") == "B":
                        a2["selected_action"] = None  # ghost
                    new_anns.append(a2)
                meta["annotations"] = new_anns
            new_ex["meta"] = meta

        return new_ex, True, "drop_B_rename_C_to_B"


# ---- run ----
rows = read_jsonl(IN_PATH)

fixed = []
changed = 0
notes = Counter()

for ex in rows:
    new_ex, did_change, note = repair_abc_instance(ex)
    fixed.append(new_ex)
    if did_change:
        changed += 1
        print(new_ex)
        print("\n")
    if note:
        notes[note] += 1

print("total:", len(rows))
print("changed:", changed)
print("notes:", notes)

write_jsonl(OUT_PATH, fixed)
print("wrote:", OUT_PATH)



total: 1952
changed: 0
notes: Counter()
wrote: unibench_skeleton.fixed_ABC.jsonl


In [2]:
import json
from collections import Counter

label_counter = Counter()

with open("unibench_skeleton.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        ls = tuple(ex.get("label_space", []))
        label_counter[ls] += 1

print(label_counter)

Counter({('A', 'B'): 1952})


# SKELETON DATA JSON 오류 검토

In [ ]:
import json

path = "roleconflict_allocation.jsonl"

bad_lines = []

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            json.loads(line)
        except Exception as e:
            bad_lines.append((i, str(e), line[:500]))

print("Broken lines:", len(bad_lines))

for i, err, snippet in bad_lines[:10]:
    print(f"\nLine {i}")
    print("Error:", err)
    print("Snippet:", snippet)

In [ ]:
def brace_check(s):
    return s.count("{") == s.count("}")

bad_brace = []

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if not brace_check(line):
            bad_brace.append((i, line[:300]))

print("Brace mismatch lines:", len(bad_brace))

for i, snippet in bad_brace[:10]:
    print(f"\nLine {i}")
    print(snippet)

In [ ]:
import json

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            print(f"\nLine {i}")
            print("Error:", e)
            print("Column:", e.colno)
            print(line)
            break